In [17]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pyarrow as pa
print(pa.__version__)
%run data_split.ipynb


26.0.0
Train: 58516 | Val: 19506 | Test: 19506


In [18]:
train_price = pd.to_numeric(train_df['price'], errors='coerce').fillna(0)
train_is_free = (train_price == 0).astype(int)
train_dlc = pd.to_numeric(train_df['dlc_count'], errors='coerce').fillna(0)
X_train = pd.DataFrame({
    "price": train_price,
    "is_free": train_is_free,
    "dlc_count": train_dlc
},
index = train_df.index)
y_train = train_df["hit"]


In [19]:
val_price = pd.to_numeric(val_df["price"], errors="coerce").fillna(0)
val_is_free = (val_price == 0).astype(int)
val_dlc = pd.to_numeric(val_df["dlc_count"], errors="coerce").fillna(0)

X_val = pd.DataFrame(
    {
        "price": val_price,
        "is_free": val_is_free,
        "dlc_count": val_dlc,
    },
    index=val_df.index,
)
y_val = val_df["hit"]


In [20]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_val)
print("Dummy Classifier Accuracy:", accuracy_score(y_val, dummy_pred))
print("Dummy Classifier Recall:", recall_score(y_val, dummy_pred))


Dummy Classifier Accuracy: 0.7469496565159438
Dummy Classifier Recall: 0.0


In [21]:
model = LogisticRegression(max_iter=3000)
model.fit(X_train, y_train)
val_pred = model.predict(X_val)
print("Logistic Regression Accuracy:", accuracy_score(y_val, val_pred))
print("Logistic Regression Recall:", recall_score(y_val, val_pred))

Logistic Regression Accuracy: 0.7539731364708295
Logistic Regression Recall: 0.04578606158833063


In [30]:
print(
    f"Accuracy: {accuracy_score(y_val, val_pred):.4f} | "
    f"Recall: {recall_score(y_val, val_pred, zero_division=0):.4f}\n"
)
print("Check:")
for col in X_train.columns:
    cols_without = [c for c in X_train.columns if c != col]
    m = LogisticRegression(max_iter=3000)
    m.fit(X_train[cols_without], y_train)
    p = m.predict(X_val[cols_without])
    print(
        f"Without {col:10} ->"
        f" Accuracy: {accuracy_score(y_val, p):.4f},"
        f" Recall: {recall_score(y_val, p):.4f}"
    )

Accuracy: 0.7540 | Recall: 0.0458

Check:
Without price      -> Accuracy: 0.7538, Recall: 0.0452
Without is_free    -> Accuracy: 0.7537, Recall: 0.0478
Without dlc_count  -> Accuracy: 0.7469, Recall: 0.0000


In [ ]:
#adding another feature to try and improve the results 
train_platforms = (
    train_df["windows"].fillna(False).astype(int)
    + train_df["mac"].fillna(False).astype(int)
    + train_df["linux"].fillna(False).astype(int)
)
val_platforms = (
    val_df["windows"].fillna(False).astype(int)
    + val_df["mac"].fillna(False).astype(int)
    + val_df["linux"].fillna(False).astype(int)
)



In [32]:
X_train_2 = pd.DataFrame(
    {
        "price": train_price,
        "is_free": train_is_free,
        "dlc_count": train_dlc,
        "platforms": train_platforms,
    },
    index=train_df.index,
)
X_val_2 = pd.DataFrame(
    {
        "price": val_price,
        "is_free": val_is_free,
        "dlc_count": val_dlc,
        "platforms": val_platforms,
    },
    index=val_df.index,
)
model_2 = LogisticRegression(max_iter=3000)
model_2.fit(X_train_2, y_train)
val_pred_2 = model_2.predict(X_val_2)
print(
    f"With platforms -> "
    f"Accuracy: {accuracy_score(y_val, val_pred_2):.4f} | "
    f"Recall: {recall_score(y_val, val_pred_2, zero_division=0):.4f}\n"
)

for col in X_train_2.columns:
    cols_without = [c for c in X_train_2.columns if c != col]
    m = LogisticRegression(max_iter=3000)
    m.fit(X_train_2[cols_without], y_train)
    p = m.predict(X_val_2[cols_without])
    print(
        f"Without {col:10} ->"
        f" Accuracy: {accuracy_score(y_val, p):.4f},"
        f" Recall: {recall_score(y_val, p):.4f}"
    )



With platforms -> Accuracy: 0.7544 | Recall: 0.0515

Without price      -> Accuracy: 0.7544, Recall: 0.0515
Without is_free    -> Accuracy: 0.7519, Recall: 0.0464
Without dlc_count  -> Accuracy: 0.7451, Recall: 0.0000
Without platforms  -> Accuracy: 0.7540, Recall: 0.0458


In [33]:
print("The price is not affecting the results, therefore I remove it for the final model.")

X_train_final = pd.DataFrame(
    {
        "is_free": train_is_free,
        "dlc_count": train_dlc,
        "platforms": train_platforms,
    },
    index=train_df.index,
)
X_val_final = pd.DataFrame(
    {
        "is_free": val_is_free,
        "dlc_count": val_dlc,  
        "platforms": val_platforms,
    },
    index=val_df.index,
)
model_final = LogisticRegression(max_iter=3000)
model_final.fit(X_train_final, y_train)
val_pred_final = model_final.predict(X_val_final)
print(
    f"Final model -> "
    f"Accuracy: {accuracy_score(y_val, val_pred_final):.4f} | "
    f"Recall: {recall_score(y_val, val_pred_final, zero_division=0):.4f}\n"
)


The price is not affecting the results, therefore I remove it for the final model.
Final model -> Accuracy: 0.7544 | Recall: 0.0515

